# 03. ベースライン実験

**対応するテキスト**: [docs/05_ベースライン実験.md](../docs/05_ベースライン実験.md)

このノートブックで行うこと:

1. ワークスペースへの接続
2. **3 本のベースライン ジョブの投入**
   - Step A: `PandaReach-v3`（学習が進むことの確認）
   - Step B: `PandaPickAndPlace-v3` + **HER**（ベースライン本体）
   - 対照: `PandaPickAndPlace-v3` **HER なし**
3. 結果の取得と比較表の作成

> ⚠ **先に [01_setup_azureml.ipynb](01_setup_azureml.ipynb) の疎通確認を完了してください。**

## 1. 接続と設定

In [ ]:
from azure.ai.ml import MLClient, command
from azure.identity import DefaultAzureCredential

# ============================================================
#  01_setup_azureml.ipynb と同じ値を入れてください
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "rl-panda-gym-env@latest"   # 最新版の環境を使う
EXPERIMENT = "rl-baseline"

TAGS = {
    "project": "rl-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
    "phase": "baseline",
}

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name, "/", ws.location)

## 2. 【重要】ベースライン条件を定義する

> **この辞書が「ベースライン条件」そのものです。**
> **改善実験は、ここから 1 要素だけを変えて比較します。**
>
> `--total-timesteps 50000` は **短時間で必ず終わる値**として設定しています。
> **学習を完成させるには足りない値**ですが、本ハンズオンの目的はモデルの完成ではなく、
> **実験を実行・記録・比較できるようになること**です。

In [ ]:
BASELINE = dict(
    algo="sac",
    reward_mode="sparse",
    total_timesteps=50_000,
    seed=0,
    learning_rate=3e-4,
    gamma=0.99,
    batch_size=256,
    buffer_size=200_000,
    learning_starts=1_000,
    eval_freq=5_000,
    n_eval_episodes=30,
    final_eval_episodes=100,
    reward_fn_version="v1",
)


def build_command(env_id: str, use_her: int, **overrides) -> str:
    """train_rl.py に渡すコマンド文字列を組み立てる。

    BASELINE の各キーは train_rl.py の引数名（アンダースコアをハイフンに置換）と
    1 対 1 で対応している。**キー名を変えると即座にジョブが失敗する。**
    """
    cfg = {**BASELINE, **overrides}
    parts = [
        "python train_rl.py",
        f"--env-id {env_id}",
        f"--use-her {use_her}",
    ]
    for key, value in cfg.items():
        parts.append(f"--{key.replace('_', '-')} {value}")
    return " ".join(parts)


def submit(env_id: str, use_her: int, display_name: str, **overrides):
    job = command(
        code="../src",
        command=build_command(env_id, use_her, **overrides),
        environment=ENV_REF,
        compute=COMPUTE_NAME,
        experiment_name=EXPERIMENT,
        display_name=display_name,
        tags={**TAGS, "env_id": env_id, "use_her": str(use_her)},
    )
    returned = ml_client.jobs.create_or_update(job)
    print(f"投入: {display_name}")
    print(f"  job name : {returned.name}")
    print(f"  studio   : {returned.studio_url}")
    return returned


print("コマンド例:")
print(" ", build_command("PandaPickAndPlace-v3", 1))

## 3. ベースライン 3 本を投入する

> ⚠ **クォータに注意。** 3 本同時に走るとノードを 3 つ使います。
> `max_instances` とクォータが足りない場合は、1 本ずつ実行してください。

In [ ]:
jobs = {}

# --- Step A: 学習が進むことの確認（安心材料） ---
#   Reach は achieved_goal が手先そのものなので、HER なしでも学習が進む
jobs["reach"] = submit(
    "PandaReach-v3", use_her=0,
    display_name="baseline_reach_sac_seed0_v1",
    total_timesteps=20_000, eval_freq=2_000,
)

# --- Step B: ベースライン本体 ---
jobs["pnp_her"] = submit(
    "PandaPickAndPlace-v3", use_her=1,
    display_name="baseline_pnp_sac-her_seed0_v1",
)

# --- 対照実験: HER の効果を測るために必須 ---
jobs["pnp_noher"] = submit(
    "PandaPickAndPlace-v3", use_her=0,
    display_name="baseline_pnp_sac-noher_seed0_v1",
)

print("\n投入した 3 本:", {k: v.name for k, v in jobs.items()})

## 4. 進捗を確認する

**studio の URL を開いて［メトリック］タブを見るのが最も分かりやすい**です。

ノートブック内で待つ場合は次のセルを実行してください（1 本ずつストリーミングします）。

In [ ]:
for key, job in jobs.items():
    status = ml_client.jobs.get(job.name).status
    print(f"{key:12s} {status:12s} {job.name}")

In [ ]:
# ログをストリーミングして完了を待つ（時間がかかります）
ml_client.jobs.stream(jobs["pnp_her"].name)

## 5. 結果を集計して比較表を作る

> ⚠ **初心者がハマる仕様**: `run.data.metrics` は **同じ名前のメトリックの最後の値しか返しません。**
> 学習曲線が欲しい場合は `MlflowClient.get_metric_history()` を使ってください。
>
> 出典: [Log metrics, parameters, and files with MLflow - Microsoft Learn](https://learn.microsoft.com/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2)

In [ ]:
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

try:
    tracking_uri = ml_client.workspaces.get(WORKSPACE_NAME).mlflow_tracking_uri
except AttributeError:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient()

KEY_METRICS = [
    "final_success_rate",
    "final_success_rate_stderr",
    "final_mean_reward",
    "final_std_reward",
    "final_mean_episode_length",
    "train_minutes",
    "video_recorded",
]
KEY_PARAMS = ["env_id", "algo", "her_effective", "reward_mode", "total_timesteps", "seed"]

rows = []
for key, job in jobs.items():
    run = mlflow.get_run(job.name)
    row = {"label": key, "job_name": job.name}
    row.update({p: run.data.params.get(p) for p in KEY_PARAMS})
    row.update({m: run.data.metrics.get(m) for m in KEY_METRICS})
    rows.append(row)

df = pd.DataFrame(rows)
df

## 6. 学習曲線を描く

**`eval_success_rate` の推移**を 3 本重ねて表示します。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for metric, ax in zip(["eval_success_rate", "eval_mean_reward"], axes):
    for key, job in jobs.items():
        try:
            hist = client.get_metric_history(job.name, metric)
        except Exception as exc:
            print(f"[WARN] {key} の {metric} を取得できません: {exc}")
            continue
        if not hist:
            continue
        hist = sorted(hist, key=lambda m: m.step)
        ax.plot([m.step for m in hist], [m.value for m in hist], marker="o", label=key)
    ax.set_xlabel("timesteps")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## 7. 【必須】評価動画をダウンロードして再生する

> **報酬ハッキングを検出する最も確実な手段は「動画を見ること」です。**
> **必ず全ての Run について再生してください。**

In [ ]:
import os

os.makedirs("downloads", exist_ok=True)

for key, job in jobs.items():
    artifacts = [a.path for a in client.list_artifacts(job.name)]
    print(f"{key}: {artifacts}")
    if "eval_video.mp4" in artifacts:
        local = client.download_artifacts(job.name, "eval_video.mp4", f"downloads/{key}")
        print(f"   → ダウンロード: {local}")
    else:
        print("   → eval_video.mp4 がありません。docs/05 のトラブルシューティング #9 を参照")

## 8. ✅ チェックリスト（ベースライン実験の完了条件）

- [ ] 3 本のジョブがすべて `Completed` になった
- [ ] 比較表（5 節の DataFrame）を作成した
- [ ] 学習曲線（6 節）を確認した
- [ ] **すべての Run の `eval_video.mp4` を再生した**
- [ ] **成功率と平均報酬を突き合わせ、報酬ハッキングの有無を判定した**
- [ ] HER あり／なしの差をワークシートに記録した
- [ ] ベースライン条件（引数・環境バージョン・ライブラリ版）を固定・記録した

→ 結果の読み解き方は [docs/06_結果の読み解き.md](../docs/06_結果の読み解き.md) を参照してください。

---

## ⚠ 本日の終了時に

**コンピューティング インスタンスを停止してください。**
クラスターは `min_instances=0` なので自動でノードが解放されます。